# Gold Layer: Feature Engineering and Model Preparation

This notebook applies strict data leakage rules, removes direct churn signals, derives tenure/duration features from dates, and constructs a clean, isolated `model_ready_data.csv`.

In [11]:
import pandas as pd
import numpy as np
from datetime import datetime
import os

In [12]:
# Load joined dataset
date_cols = ['agreement_start_date', 'agreement_end_date']
df = pd.read_csv('../../data/02_intermediate/joined_data.csv', parse_dates=date_cols, on_bad_lines='skip')

print(f"Loaded dataset shape: {df.shape}")

Loaded dataset shape: (21526, 40)


In [13]:
def build_model_ready_set(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    # Rename to maintain sanity with requested names
    if 'branch' in df.columns and 'branch_bob' not in df.columns:
        df.rename(columns={'branch': 'branch_bob'}, inplace=True)
    if 'agreement_end_date' in df.columns and 'agreement_end_date_bob' not in df.columns:
        df.rename(columns={'agreement_end_date': 'agreement_end_date_bob'}, inplace=True)
    if 'churn_category' in df.columns and 'churn' not in df.columns:
        df.rename(columns={'churn_category': 'churn'}, inplace=True)
        
    # 1. DROP (STRICT DATA LEAKAGE)
    leakage_cols = [
        'pull_van', 'new_van', 'van', 'number_of_repair_cases', 'number_of_overdueservices',
        'case_title', 'case_type', 'risk', 'current_status', 'case_origin',
        'case_creation_date', 'resolved_date', 'registered_date', 'expected_pull_date',
        'resolution_status'
    ]
    
    # 2. DROP (DIRECT CHURN SIGNAL / PROXY)
    proxy_cols = ['lost_agreements']
    
    # 3. DROP (IDENTIFIERS / DUPLICATES)
    id_cols = ['account_number', 'customer_account_number']
    
    # 4. DROP (HIGH RISK / CONTEXTUAL)
    high_risk_cols = ['pull_type', 'customer_tier', 'companysize']
    
    all_drops = leakage_cols + proxy_cols + id_cols + high_risk_cols
    existing_drops = [c for c in all_drops if c in df.columns]
    df.drop(columns=existing_drops, inplace=True)
    
    # Convert Dates -> Features
    if 'agreement_start_date' in df.columns and 'agreement_end_date_bob' in df.columns:
        # Coerce to datetime explicitely to prevent string array mismatch exceptions
        df['agreement_start_date'] = pd.to_datetime(df['agreement_start_date'], errors='coerce')
        df['agreement_end_date_bob'] = pd.to_datetime(df['agreement_end_date_bob'], errors='coerce')
        
        # Use a fixed reference point ('today')
        today = pd.to_datetime('today').normalize()
        
        df['tenure_days'] = (today - df['agreement_start_date']).dt.days
        df['contract_duration'] = (df['agreement_end_date_bob'] - df['agreement_start_date']).dt.days
        
        # Drop original dates
        df.drop(columns=['agreement_start_date', 'agreement_end_date_bob'], inplace=True)
    
    # 5. KEEP (CORE FEATURES ONLY)
    final_keep_columns = [
        'total_agreements', 'number_of_contracts', 'machines',
        'product_bob', 'fee_bob', 'total_bob', 'service_interval', 'unit_amount',
        'billing_interval', 'tenure_days', 'contract_duration', 'company_sizing',
        'branch_bob', 'renewal_type', 'agreement_type', 'line_of_business',
        'billing_period', 'churn'
    ]
    
    final_cols = [c for c in final_keep_columns if c in df.columns]
    df = df[final_cols]
    
    # Handle any lingering missing values (Median/Mode)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    categorical_cols = df.select_dtypes(exclude=[np.number]).columns
    
    for col in numeric_cols:
        df[col] = df[col].fillna(df[col].median())
        
    for col in categorical_cols:
        mode_s = df[col].mode()
        if not mode_s.empty:
            df[col] = df[col].fillna(mode_s.iloc[0])
        else:
            df[col] = df[col].fillna('Unknown')
            
    # Factorize categorical for baseline models
    for col in categorical_cols:
        df[col] = pd.factorize(df[col])[0]
        
    return df

model_df = build_model_ready_set(df)

os.makedirs('../../data/03_processed', exist_ok=True)
model_df.to_csv('../../data/03_processed/model_ready_data.csv', index=False)

print("Model-ready dataset saved to ../../data/03_processed/model_ready_data.csv")
print(f"\nFinal dataset shape: {model_df.shape}")
print("\nFinal Columns in Order:", model_df.columns.tolist())
if 'churn' in model_df.columns:
    print("\nTarget distribution:")
    print(model_df['churn'].value_counts())


Model-ready dataset saved to ../../data/03_processed/model_ready_data.csv

Final dataset shape: (21526, 18)

Final Columns in Order: ['total_agreements', 'number_of_contracts', 'machines', 'product_bob', 'fee_bob', 'total_bob', 'service_interval', 'unit_amount', 'billing_interval', 'tenure_days', 'contract_duration', 'company_sizing', 'branch_bob', 'renewal_type', 'agreement_type', 'line_of_business', 'billing_period', 'churn']

Target distribution:
churn
2    16233
0     4226
1     1067
Name: count, dtype: int64


In [14]:
import pandas as pd
import numpy as np

# Load the model-ready dataset
df = pd.read_csv('../../data/03_processed/model_ready_data.csv')
print(f"Original dataset shape: {df.shape}")

# Feature Engineering
print("\n--- Creating Engineered Features ---")

# 1. Duration to Tenure Ratio
df['duration_to_tenure_ratio'] = np.where(df['contract_duration'] > 0, df['tenure_days'] / df['contract_duration'], 0)


# Add missing alias columns requested
if 'total_bob' in df.columns:
    df['total_bob_sum'] = df['total_bob']

if 'total_agreements' in df.columns:
    df['agreement_count'] = df['total_agreements']

# Save the updated dataset
df.to_csv('../../data/03_processed/model_ready_data.csv', index=False)
print(f"\nFinal dataset shape after feature engineering: {df.shape}")
print(f"Final Dataset Columns: {df.columns.tolist()}")
print("Updated model_ready_data.csv successfully!")


Original dataset shape: (21526, 18)

--- Creating Engineered Features ---

Final dataset shape after feature engineering: (21526, 21)
Final Dataset Columns: ['total_agreements', 'number_of_contracts', 'machines', 'product_bob', 'fee_bob', 'total_bob', 'service_interval', 'unit_amount', 'billing_interval', 'tenure_days', 'contract_duration', 'company_sizing', 'branch_bob', 'renewal_type', 'agreement_type', 'line_of_business', 'billing_period', 'churn', 'duration_to_tenure_ratio', 'total_bob_sum', 'agreement_count']
Updated model_ready_data.csv successfully!
